# Task 10: transcribe the validation data (E387 wind-tunnel polars, Warren-12 reference)

The Claude Code sandbox cannot reach the UIUC airfoil data site or MIT, so the two validation data
sets of `docs/specs/task10_drone_sizing.md` section 7 are transcribed here, on a machine with open
internet, from files you download yourself. Nothing is typed from memory: every number in the YAML
comes from a file whose name and sha256 are written next to it.

**What to download first** (on this machine, then upload below):
1. UIUC Low-Speed Airfoil Tests, Eppler E387 polars at Re 100 000, 200 000 and 300 000, from the UIUC
   airfoil data site (Selig et al., *Summary of Low-Speed Airfoil Data*, Volume 1, 1995). Download the
   plain-text polar files (columns alpha, Cl, Cd; other columns are ignored). Note the volume and the
   page or file name.
2. The source that states the Warren-12 reference derivatives (CL_alpha 2.743 /rad, Cm_alpha -3.10
   /rad) together with its moment reference point: the Tornado user's guide (Melin, KTH 2000) or
   the VSPAERO validation case. Read the moment reference x off it and type it in cell 4 with the
   citation; this is the one value entered by hand, and the cell records who read it and where.

## 1. Repository

In [ ]:
COMMIT = "main"
import os, subprocess, sys, hashlib, json, re
if not os.path.isdir("/content/ai-engineering"):
    !git clone -q https://github.com/emarinhoss/ai-engineering.git /content/ai-engineering
%cd /content/ai-engineering
!git fetch -q --all && git checkout -q {COMMIT} && git pull -q --ff-only
!pip install -q pyyaml
import yaml
HEAD = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("commit", HEAD)

## 2. Upload the E387 polar files
One file per Reynolds number. The parser keeps every line with at least three numeric columns and
reads them as alpha (deg), Cl, Cd; it prints what it found so you can check the columns against the
file header before anything is written.

In [ ]:
from google.colab import files
uploaded = files.upload()      # pick the three E387 polar text files
def parse_polar(text):
    rows = []
    for line in text.splitlines():
        parts = line.replace(",", " ").split()
        try:
            vals = [float(v) for v in parts[:3]]
        except ValueError:
            continue
        if len(vals) == 3:
            rows.append(vals)
    return rows
polars = {}
for name, blob in uploaded.items():
    text = blob.decode("utf-8", errors="replace")
    rows = parse_polar(text)
    m = re.search(r"(\d{2,3})\s*[kK]|Re\D*(\d{5,6})|(\d{6})", name + " " + text[:300])
    print(f"{name}: {len(rows)} numeric rows; first {rows[:2]}; sha256 {hashlib.sha256(blob).hexdigest()[:16]}")
    polars[name] = {"rows": rows, "sha256": hashlib.sha256(blob).hexdigest(), "header": text[:300]}

## 3. Assign the Reynolds number of each file and write `data/validation/e387_uiuc.yaml`
Edit `RE_OF` so each file name maps to its Reynolds number as the file header states it.

In [ ]:
RE_OF = {name: None for name in polars}      # e.g. {"e387_100k.txt": 1e5, ...}; fill in from the headers printed above
print(RE_OF)
assert all(v for v in RE_OF.values()), "fill in RE_OF first"
doc = {"set": "validation/e387_uiuc",
       "source": "Selig, Guglielmo, Broeren and Giguere, Summary of Low-Speed Airfoil Data, Volume 1, SoarTech "
                 "Publications, 1995; files downloaded from the UIUC Low-Speed Airfoil Tests site "
                 "(m-selig.ae.illinois.edu/uiuc_lsat). Transcribed by this notebook; one entry per file with its sha256.",
       "retrieval": "done", "retrieved_on": __import__("datetime").date.today().isoformat(), "retrieved_by": "Eder (Colab)",
       "polars": []}
for name, p in polars.items():
    rows = sorted(p["rows"])
    doc["polars"].append({"reynolds": float(RE_OF[name]), "file": name, "sha256": p["sha256"],
                          "alpha_deg": [r[0] for r in rows], "cl": [r[1] for r in rows], "cd": [r[2] for r in rows]})
path = "data/validation/e387_uiuc.yaml"
with open(path) as f:                      # keep the committed coordinates block
    old = yaml.safe_load(f) or {}
if "coordinates" in old:
    doc["coordinates"] = old["coordinates"]
with open(path, "w") as f:
    yaml.safe_dump(doc, f, sort_keys=False, width=100)
print(open(path).read()[:1200])

## 4. The Warren-12 moment reference
Type the moment reference x (origin at the root leading edge, metres in the same units as the chords)
exactly as the source states it, and the citation. The cell updates only that field and the retrieval
fields of `data/validation/warren12.yaml`.

In [ ]:
MOMENT_REFERENCE_X = None      # e.g. 0.0 if the source takes moments about the apex; read it off the document
CITATION = ""                  # document, section, page
assert MOMENT_REFERENCE_X is not None and CITATION, "fill both in from the source"
path = "data/validation/warren12.yaml"
with open(path) as f:
    w = yaml.safe_load(f)
w["warren12"]["moment_reference_x"] = float(MOMENT_REFERENCE_X)
w["warren12"]["moment_reference_source"] = CITATION
w["warren12"]["retrieval"] = "done"
with open(path, "w") as f:
    yaml.safe_dump(w, f, sort_keys=False, width=100)
print(open(path).read())

## 5. Hand the two files to the repository
Downloads both YAMLs; put them in `data/validation/` on a checkout and commit them (or paste their
sha256 into the Claude Code session so the copies there can be checked). Then
`python -m pytest tests/task10/test_ledger.py tests/task10/test_vlm.py -q` runs the two skipped checks.

In [ ]:
for p in ("data/validation/e387_uiuc.yaml", "data/validation/warren12.yaml"):
    print(p, hashlib.sha256(open(p, "rb").read()).hexdigest())
    files.download(p)